# 01. Preprocesamiento

Etapa 02: Carga de datos crudos, limpieza, conversión de moneda y preparación para análisis.

**Fuente:** `data/raw/clasificados/revolico/anuncios.csv`  
**Salidas:** `data/interim/anuncios_limpios.csv`, `data/processed/mercado_havana.csv`

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.chdir('..')
import pandas as pd
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
TASA_CUP_USD = 710.0
RAW = 'data/raw/clasificados/revolico/anuncios.csv'
df = pd.read_csv(RAW, encoding='utf-8-sig')
df.head()

In [ ]:
# Limpieza básica
from src.utils import limpieza as L
import numpy as np
df['precio_original'] = pd.to_numeric(df.get('precio_original', df.get('precio', np.nan)), errors='coerce')
df['moneda'] = df.get('moneda','USD').astype(str).str.upper()
df['precio_usd'] = df['precio_original'].apply(lambda x: x/TASA_CUP_USD if pd.notna(x) else np.nan)
mask_cup = df['moneda'].str.contains('CUP')
df.loc[mask_cup & df['precio_original'].notna(), 'precio_usd'] = df.loc[mask_cup & df['precio_original'].notna(), 'precio_original']/TASA_CUP_USD
df.loc[~mask_cup & df['precio_original'].notna(), 'precio_usd'] = df.loc[~mask_cup & df['precio_original'].notna(), 'precio_original']
df['anio'] = pd.to_numeric(df.get('anio'), errors='coerce').fillna(0).astype(int)
df['marca'] = df.get('marca','').apply(L.clean_marca)
df['municipio'] = df.get('municipio', df.get('ubicacion','')).fillna('')
df['tipo_fuente'] = 'clasificados'; df['mercado'] = 'informal'
df.to_csv('data/interim/anuncios_limpios.csv', index=False, encoding='utf-8-sig')
df_proc = df.copy()
df_proc.to_csv('data/processed/mercado_havana.csv', index=False, encoding='utf-8-sig')
print('ok', len(df))